# RigTech · runner_colab_continuacao

Notebook de **continuação** — pensado para retomar o trabalho em uma **conta Google nova** sem retrabalho.

Diferença para `runner_colab.ipynb`:
- **Não reconverte** o dataset: espera `work_processed.tar.gz` (1.1 GB) já no Drive novo.
- **Restaura o baseline v1** se `runs/v1_baseline_colab_*/train/weights/best.pt` estiver no Drive — evita retreinar o baseline.
- Cai direto no que estava bloqueado: **LOO (Fase 5)** ou **próximo ciclo (v2)**.

## Pré-requisitos no Drive novo

Estrutura mínima esperada em `MyDrive/rigtech-weed-cycle/`:
```
rigtech-weed-cycle/
  work_processed.tar.gz        # OBRIGATÓRIO — dataset YOLO-seg já convertido
  runs/                        # OPCIONAL — se tiver, restaura baseline v1
    history.json
    v1_baseline_colab_*/train/weights/best.pt
```

Se **não** tiver o tarball, use `runner_colab.ipynb` (roda o pipeline inteiro do zero a partir de `DaninhasTreinoClientes/`).

Runtime → Change runtime type → **GPU A100** (T4 dobra o tempo).

## 1. Repo e deps

In [ ]:
!git clone https://github.com/maluquintela/rigtech-weed-cycle.git
%cd rigtech-weed-cycle

In [ ]:
!pip install -q ultralytics==8.4.115 rasterio shapely pyproj pillow pyyaml requests

## 2. Drive → disco local

Nunca trabalhar direto no Drive montado — latência mata. Sempre copiar para `/content/` primeiro.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Raiz do projeto no Drive novo. Ajustar se você usar outro nome.
DRIVE_ROOT = '/content/drive/MyDrive/rigtech-weed-cycle'
!ls -la "{DRIVE_ROOT}"

## 3. Restaurar dataset processado

Extrai `work_processed.tar.gz` direto para `/content/rigtech-weed-cycle/work/`. Isso repõe `work/live`, `work/golden`, `work/versions/v1/` — tudo que a Fase 1-3 geraria.

In [ ]:
import os
TAR = f'{DRIVE_ROOT}/work_processed.tar.gz'
assert os.path.exists(TAR), f'Faltando {TAR}. Suba o tarball ou use runner_colab.ipynb (pipeline do zero).'
!ls -lh "{TAR}"
# Copia para disco local primeiro (leitura do Drive é lenta)
!cp "{TAR}" /content/work_processed.tar.gz
!tar -xzf /content/work_processed.tar.gz -C .
!ls -la work/

In [ ]:
# Sanidade: contagem de tiles por split
!echo '--- live/train ---'; ls work/live/images/train 2>/dev/null | wc -l
!echo '--- live/val ---';   ls work/live/images/val   2>/dev/null | wc -l
!echo '--- golden ---';     ls work/golden/images     2>/dev/null | wc -l
!echo '--- versions ---';   ls work/versions          2>/dev/null

## 4. Restaurar baseline v1 (se houver)

Se você já tem `runs/` no Drive, sincroniza pra `work/runs/` — assim `history.json` e `best.pt` do baseline ficam disponíveis. Se não tiver, pula (vai treinar o baseline nesta sessão).

In [ ]:
import os, subprocess
RUNS_DRIVE = f'{DRIVE_ROOT}/runs'
if os.path.isdir(RUNS_DRIVE):
    os.makedirs('work/runs', exist_ok=True)
    subprocess.run(['rsync', '-av', RUNS_DRIVE + '/', 'work/runs/'], check=True)
    print('runs restauradas do Drive.')
    !ls work/runs/
else:
    print(f'{RUNS_DRIVE} não existe — nenhum baseline pra restaurar.')

In [ ]:
# Se o baseline v1 estiver presente, mostra as métricas registradas
import json, os
hist_path = 'work/runs/history.json'
if os.path.exists(hist_path):
    with open(hist_path) as f:
        hist = json.load(f)
    for run in hist[-3:]:
        print(run.get('version'), run.get('tag'), '→', run.get('metrics', {}))
else:
    print('sem history.json — vai treinar baseline do zero.')

## 5. Linear API key

O hook do `train_eval.py` posta o run em Linear automaticamente se `LINEAR_API_KEY` estiver setada. Cole a chave abaixo (não faz commit — vive só na sessão).

In [ ]:
import os
from getpass import getpass
os.environ['LINEAR_API_KEY'] = getpass('LINEAR_API_KEY (Enter pra pular): ') or ''
print('key setada' if os.environ['LINEAR_API_KEY'] else 'sem key — Linear não vai ser atualizado')

## 6. (Opcional) treinar baseline v1 — só se não foi restaurado

Se `work/runs/v1_baseline_colab_*/train/weights/best.pt` já existe, **pule esta célula**.

In [ ]:
import glob
if not glob.glob('work/runs/v1_baseline_colab_*/train/weights/best.pt'):
    !python -m src.train_eval --version v1 --tag baseline_colab \
        --device 0 --batch 16 --imgsz 1024 --amp --epochs 100
else:
    print('baseline v1 já presente — pulei o treino.')

## 7. Fase 5 — Leave-one-out por talhão (o bloqueio atual)

5 treinos (1 por talhão como val). Diagnóstica se o gap de generalização é específico de Giasa ou universal.

**Custo**: ~1.6h/fold × 5 = ~8h em T4 (menos em A100). `--skip-existing` deixa retomar de onde parou se a sessão morrer.

In [ ]:
!git pull  # pega qualquer coisa nova em src/

In [ ]:
!python -m src.loo_train \
    --talhoes Celso01 CelsoSTE2 Flaviano01 DoisRiosFlaviano Giasa \
    --device 0 --batch 16 --imgsz 1024 --amp --epochs 100 --skip-existing

## 8. Backup incremental → Drive

Rodar depois de cada fold, ou no fim. Só sobe artefatos leves (weights, plots, csv, json) — nunca o dataset materializado.

In [ ]:
!mkdir -p "{DRIVE_ROOT}/runs"
!cp work/runs/history.json "{DRIVE_ROOT}/runs/" 2>/dev/null
!rsync -av --include='*.pt' --include='*.png' --include='*.csv' --include='*.yaml' \
  --include='*/' --exclude='*' work/runs/ "{DRIVE_ROOT}/runs/"

## 9. Ciclo v2 — quando quiser avançar

Fluxo: detectar suspeitas no golden → postar em Linear → anotador corrige → snapshot v2 → treinar → comparar.

In [ ]:
# 9a. Detectar tiles suspeitas (usa best.pt do último baseline)
!python -m src.suspects

In [ ]:
# 9b. Postar suspeitas em Linear com preview
!python -m src.link_suspects_to_linear --limit 40

In [ ]:
# 9c. Após anotação humana concluída em work/live/, criar snapshot v2 e treinar
# !python -m src.snapshot --note 'v2: correções do ciclo 1 (RIG-XXX..RIG-YYY)'
# !python -m src.train_eval --version v2 --tag pos_ciclo1 --device 0 --batch 16 --imgsz 1024 --amp --epochs 100

In [ ]:
# 9d. Relatório comparativo do ciclo
# !python -m src.cycle_report --from v1 --to v2